# Day 40 — Practice 2 & 3: AdventureWorks → HDFS Parquet + Partisi

**Tujuan:** Extract data dari PostgreSQL AdventureWorks, simpan ke HDFS sebagai Parquet, dan terapkan partisi.

Alur:
```
PostgreSQL (AdventureWorks)  →  Spark  →  HDFS Parquet (Data Lake)
                                                ↓
                                    /datalake/raw/sales_orders/
                                        year=2011/
                                        year=2012/
                                        year=2013/
                                        year=2014/
```

**Konsep yang dipraktikkan:**
- JDBC read dari PostgreSQL
- Tulis ke HDFS sebagai Parquet (columnar format)
- Partisi berdasarkan kolom tertentu
- Manfaat partisi: **partition pruning** saat query

## Setup SparkSession

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AdventureWorks-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

JDBC_URL = "jdbc:postgresql://adventureworks-postgres:5432/postgres"
JDBC_PROPS = {
    "user": "postgres",
    "password": "My_password1",
    "driver": "org.postgresql.Driver"
}
HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table, query=None):
    # Pakai double quotes untuk case-sensitive schema/table
    src = query if query else f'"{schema}"."{table}"'
    return spark.read.jdbc(url=JDBC_URL, table=src, properties=JDBC_PROPS)

print(f'Spark {spark.version} ready')

Spark 3.5.0 ready


## 1. Eksplorasi Tabel AdventureWorks

In [2]:
# Lihat semua tabel yang tersedia
df_tables = read_pg(None, None, query="""
    (SELECT table_schema, table_name, 
            pg_size_pretty(pg_total_relation_size(quote_ident(table_schema)||'.'||quote_ident(table_name))) AS size
     FROM information_schema.tables
     WHERE table_type = 'BASE TABLE'
       AND table_schema NOT IN ('pg_catalog','information_schema')
     ORDER BY table_schema, table_name) t
""")

print(f'Total tabel: {df_tables.count()}')
df_tables.show(30, truncate=False)

Total tabel: 71
+--------------+-------------------------+-------+
|table_schema  |table_name               |size   |
+--------------+-------------------------+-------+
|HumanResources|Department               |48 kB  |
|HumanResources|Employee                 |216 kB |
|HumanResources|EmployeeDepartmentHistory|104 kB |
|HumanResources|EmployeePayHistory       |80 kB  |
|HumanResources|JobCandidate             |128 kB |
|HumanResources|Shift                    |56 kB  |
|Person        |Address                  |6168 kB|
|Person        |AddressType              |56 kB  |
|Person        |BusinessEntity           |2576 kB|
|Person        |BusinessEntityAddress    |3544 kB|
|Person        |BusinessEntityContact    |256 kB |
|Person        |ContactType              |40 kB  |
|Person        |CountryRegion            |72 kB  |
|Person        |EmailAddress             |3584 kB|
|Person        |Password                 |2776 kB|
|Person        |Person                   |16 MB  |
|Person        

## 2. Extract Semua Tabel ke HDFS (Tanpa Partisi)

Untuk tabel referensi / dimensi yang tidak terlalu besar, simpan langsung tanpa partisi.

In [3]:
# Daftar tabel dimensi / referensi
dim_tables = [
    ('Production', 'Product'),
    ('Production', 'ProductCategory'),
    ('Production', 'ProductSubcategory'),
    ('Person',     'Person'),
    ('HumanResources', 'Department'),
    ('HumanResources', 'Employee'),
    ('Sales',      'Customer'),
    ('Sales',      'SalesTerritory'),
]

results = []
for schema, table in dim_tables:
    df = read_pg(schema, table)
    row_count = df.count()
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    
    df.write \
      .mode('overwrite') \
      .parquet(hdfs_path)
    
    results.append((f'{schema}.{table}', row_count, hdfs_path))
    print(f'✓ {schema}.{table:30s} → {row_count:6,} rows → HDFS')

print('\nSemua tabel dimensi berhasil disimpan ke HDFS!')

✓ Production.Product                        →    504 rows → HDFS
✓ Production.ProductCategory                →      4 rows → HDFS
✓ Production.ProductSubcategory             →     37 rows → HDFS
✓ Person.Person                         → 19,972 rows → HDFS
✓ HumanResources.Department                     →     16 rows → HDFS
✓ HumanResources.Employee                       →    290 rows → HDFS
✓ Sales.Customer                       → 19,820 rows → HDFS
✓ Sales.SalesTerritory                 →     10 rows → HDFS

Semua tabel dimensi berhasil disimpan ke HDFS!


## 3. Extract Tabel Fakta dengan Partisi

Tabel fakta (transaksi) biasanya sangat besar. Partisi berdasarkan `year` dan `month`
sangat penting agar query lebih cepat (partition pruning).

### 3.1 Sales Order Header — Partisi by Year

In [4]:
# Baca sales order header
df_orders = read_pg('Sales', 'SalesOrderHeader')

print('=== Schema SalesOrderHeader ===')
df_orders.printSchema()

print(f'\nTotal rows: {df_orders.count():,}')
print('\nDistribusi per tahun:')
df_orders.withColumn('year', F.year('orderdate')) \
         .groupBy('year').count() \
         .orderBy('year').show()

=== Schema SalesOrderHeader ===
root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: short (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: short (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddressID: integer (nullable = true)
 |-- ShipMethodID: integer (nullable = true)
 |-- CreditCardID: integer (nullable = true)
 |-- CreditCardApprovalCode: string (nullable = true)
 |-- CurrencyRateID: integer (nullable = true)
 |-- SubTotal: decimal(19,4) (nullable = true)
 |-- TaxAmt: decimal(19,4) (nullable = true)
 |-

In [5]:
# Tambahkan kolom partisi: year dan month
df_orders_partitioned = df_orders \
    .withColumn('order_year',  F.year('orderdate')) \
    .withColumn('order_month', F.month('orderdate'))

# Simpan ke HDFS dengan partisi year, month
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

df_orders_partitioned.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(orders_path)

print(f'✓ SalesOrderHeader disimpan ke HDFS dengan partisi order_year/order_month')
print(f'  Path: {orders_path}')

✓ SalesOrderHeader disimpan ke HDFS dengan partisi order_year/order_month
  Path: hdfs://namenode:9000/datalake/raw/Sales/SalesOrderHeader


In [6]:
# Lihat struktur partisi yang terbentuk di HDFS
import subprocess

print('=== Struktur partisi di HDFS ===')
result = subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -ls -R /datalake/raw/Sales/SalesOrderHeader | head -30',
    shell=True, capture_output=True, text=True
)
print(result.stdout)

print('\n=== Ukuran per partisi ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw/Sales/SalesOrderHeader',
    shell=True
)

=== Struktur partisi di HDFS ===


=== Ukuran per partisi ===


CompletedProcess(args='docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw/Sales/SalesOrderHeader', returncode=127)

### 3.2 Sales Order Detail — Partisi by Year

In [7]:
# Order detail — join dengan header untuk ambil tanggal
df_detail = read_pg('Sales', 'SalesOrderDetail')

# Join dengan header untuk dapat order_year
df_detail_with_year = df_detail \
    .join(df_orders.select('salesorderid', 'orderdate'), 'salesorderid') \
    .withColumn('order_year', F.year('orderdate')) \
    .drop('orderdate')

print(f'Total order detail rows: {df_detail_with_year.count():,}')

detail_path = f'{HDFS_BASE}/Sales/SalesOrderDetail'
df_detail_with_year.write \
    .mode('overwrite') \
    .partitionBy('order_year') \
    .parquet(detail_path)

print(f'✓ salesorderdetail disimpan dengan partisi order_year')

Total order detail rows: 121,317
✓ salesorderdetail disimpan dengan partisi order_year


## 4. Demonstrasi Partition Pruning

Kenapa partisi itu penting? Tanpa partisi → Spark scan **semua file**.
Dengan partisi → Spark hanya scan **folder yang relevan**.

In [8]:
# Baca tabel yang sudah dipartisi
df_hdfs_orders = spark.read.parquet(orders_path)

print('=== Schema setelah baca dari HDFS ===')
df_hdfs_orders.printSchema()
print(f'Total rows: {df_hdfs_orders.count():,}')

=== Schema setelah baca dari HDFS ===
root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: short (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: short (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddressID: integer (nullable = true)
 |-- ShipMethodID: integer (nullable = true)
 |-- CreditCardID: integer (nullable = true)
 |-- CreditCardApprovalCode: string (nullable = true)
 |-- CurrencyRateID: integer (nullable = true)
 |-- SubTotal: decimal(19,4) (nullable = true)
 |-- TaxAmt: decimal(19,4) (nullable = tru

In [9]:
# Query DENGAN filter partisi (CEPAT — hanya baca folder order_year=2013)
print('=== Query dengan filter partisi (order_year=2013) ===')
df_2013 = df_hdfs_orders.filter(F.col('order_year') == 2013)
print(f'Rows tahun 2013: {df_2013.count():,}')

# Lihat physical plan — cek partition filters
print('\n=== Physical Plan (perhatikan PartitionFilters) ===')
df_2013.explain()

=== Query dengan filter partisi (order_year=2013) ===
Rows tahun 2013: 14,182

=== Physical Plan (perhatikan PartitionFilters) ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [SalesOrderID#740,RevisionNumber#741,OrderDate#742,DueDate#743,ShipDate#744,Status#745,OnlineOrderFlag#746,SalesOrderNumber#747,PurchaseOrderNumber#748,AccountNumber#749,CustomerID#750,SalesPersonID#751,TerritoryID#752,BillToAddressID#753,ShipToAddressID#754,ShipMethodID#755,CreditCardID#756,CreditCardApprovalCode#757,CurrencyRateID#758,SubTotal#759,TaxAmt#760,Freight#761,TotalDue#762,Comment#763,... 4 more fields] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/datalake/raw/Sales/SalesOrderHeader], PartitionFilters: [isnotnull(order_year#766), (order_year#766 = 2013)], PushedFilters: [], ReadSchema: struct<SalesOrderID:int,RevisionNumber:smallint,OrderDate:timestamp,DueDate:timestamp,ShipDate:ti...




In [10]:
# Query Q1 2014 saja
df_q1_2014 = df_hdfs_orders.filter(
    (F.col('order_year') == 2014) & (F.col('order_month') <= 3)
)
print(f'Orders Q1 2014: {df_q1_2014.count():,}')

df_q1_2014.select('salesorderid', 'orderdate', 'subtotal', 'order_year', 'order_month').show(5)

Orders Q1 2014: 6,296
+------------+-------------------+----------+----------+-----------+
|salesorderid|          orderdate|  subtotal|order_year|order_month|
+------------+-------------------+----------+----------+-----------+
|       67857|2014-03-08 00:00:00|   99.5700|      2014|          3|
|       68828|2014-03-23 00:00:00|   27.7700|      2014|          3|
|       67260|2014-03-01 00:00:00|39450.7848|      2014|          3|
|       67261|2014-03-01 00:00:00|29064.3900|      2014|          3|
|       67262|2014-03-01 00:00:00| 7501.7220|      2014|          3|
+------------+-------------------+----------+----------+-----------+
only showing top 5 rows



## 5. Ringkasan Semua File di HDFS

In [11]:
print('=== Semua file di /datalake/raw ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw',
    shell=True
)

print('\n=== Total ukuran datalake ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -s -h /datalake',
    shell=True
)

=== Semua file di /datalake/raw ===

=== Total ukuran datalake ===


CompletedProcess(args='docker exec hadoop-namenode hdfs dfs -du -s -h /datalake', returncode=127)

## Summary

Yang sudah dipraktikkan:
- ✅ Extract semua tabel dari PostgreSQL AdventureWorks via JDBC
- ✅ Simpan ke HDFS sebagai Parquet (compressed dengan Snappy)
- ✅ Partisi tabel fakta berdasarkan `order_year` dan `order_month`
- ✅ Demonstrasi **partition pruning** — Spark hanya baca partisi yang dibutuhkan

**Data Lake Sekarang:**
```
/datalake/raw/
  production/product/
  production/productcategory/
  person/person/
  humanresources/employee/
  sales/salesorderheader/     ← partisi order_year/order_month
  sales/salesorderdetail/     ← partisi order_year
  ...
```

**Next →** `03_hdfs_to_hive.ipynb` — Buat Hive External Table di atas data HDFS ini.